# Student Payment Data Cleaning and Reconciliation

**Tools:** Python, Pandas, Jupyter Notebook, Excel  
**Project type:** Data cleaning, data integration, data quality checks

## Project overview
This project consolidates student/internship records and payment information from six workbook sheets into a consistent master dataset. The workflow focuses on standardising columns and text values, parsing dates and amounts, matching records across sheets, aggregating payment transactions, reconciling collections against installment entries, and checking duplicates and missing values.

> **Privacy note:** The source workbook contains student-level information. It is intentionally not included in this repository. Add an authorised, anonymised copy locally before running the notebook. Do not commit names, phone numbers, student IDs, payment records, or other personal information.

## 1. Set up the environment

Install dependencies from `requirements.txt`. Keep the source workbook in `data/raw/` and use the expected filename below, or update `SOURCE_FILE`.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_colwidth", 80)

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
SOURCE_FILE = RAW_DIR / "inetz_worksheet.xlsx"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

if not SOURCE_FILE.exists():
    raise FileNotFoundError(
        f"Source workbook not found: {SOURCE_FILE}\n"
        "Place an authorised copy in data/raw/ or update SOURCE_FILE."
    )

## 2. Load the workbook sheets

The original workflow used six sheets: `Internship`, `Registered Students`, `Payment Entries`, `Student Summary`, `Registered Intern Details`, and `Course`. Header rows differ by sheet, so the `header` settings below mirror the workbook layout observed during the project. If the source workbook changes, verify these row positions before proceeding.

In [ ]:
SHEETS = {
    "internship": ("Internship", [2, 3]),
    "registered_students": ("Registered Students", 3),
    "payment_entries": ("Payment Entries", 3),
    "student_summary": ("Student Summary", 3),
    "registered_intern_details": ("Registered Intern Details", 0),
    "course": ("Course", 3),
}

raw = {
    key: pd.read_excel(SOURCE_FILE, sheet_name=sheet, header=header)
    for key, (sheet, header) in SHEETS.items()
}

for name, frame in raw.items():
    print(f"{name:28s} rows={frame.shape[0]:>5} columns={frame.shape[1]:>3}")

## 3. Inspect data quality before changing values

Review dimensions, column names, missing values, and duplicate rows. Avoid printing or committing full student-level records in public notebooks.

In [ ]:
def quality_report(df: pd.DataFrame, name: str) -> pd.DataFrame:
    return pd.DataFrame({
        "dataset": name,
        "column": df.columns.astype(str),
        "dtype": [str(dtype) for dtype in df.dtypes],
        "missing_count": df.isna().sum().to_numpy(),
        "missing_pct": (df.isna().mean().mul(100).round(2)).to_numpy(),
        "unique_values": df.nunique(dropna=True).to_numpy(),
    })

quality = pd.concat(
    [quality_report(frame, name) for name, frame in raw.items()],
    ignore_index=True
)
quality.head(30)

## 4. Standardise column labels and remove blank rows

This helper trims column labels and removes rows that are entirely empty. Sheet-specific column renaming is handled separately because the original workbook contains different header structures.

In [ ]:
def clean_frame(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [
        str(col).strip().replace("\\n", " ")
        for col in out.columns
    ]
    out = out.dropna(how="all").reset_index(drop=True)
    return out

data = {name: clean_frame(frame) for name, frame in raw.items()}

for name, frame in data.items():
    print(name, frame.shape)

## 5. Normalise text fields and data types

Text fields are trimmed and empty strings are converted to missing values. Phone numbers and student identifiers should remain text so leading zeroes are not lost. Date and monetary columns should be converted only after checking their actual column labels in the source workbook.

In [ ]:
def clean_text_columns(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    out = df.copy()
    for col in columns:
        if col in out.columns:
            out[col] = (
                out[col].astype("string")
                .str.strip()
                .replace("", pd.NA)
            )
    return out

def convert_dates(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    out = df.copy()
    for col in columns:
        if col in out.columns:
            out[col] = pd.to_datetime(out[col], errors="coerce", dayfirst=True)
    return out

def convert_amounts(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    out = df.copy()
    for col in columns:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    return out

## 6. Prepare the internship and registration tables

The source notebook standardised internship columns, removed blank/header rows, cleaned text fields, and converted dates and fee amounts. Because source workbook headers can vary, check the displayed column list and adjust the mappings if needed.

In [ ]:
# Inspect before applying schema-specific transformations.
for name in ["internship", "registered_students", "registered_intern_details", "course"]:
    print(f"\\n{name}:")
    print(data[name].columns.tolist())

In [ ]:
# Standardise known labels from the original workbook.
# These mappings are deliberately explicit so changes to the source schema are visible.
data["registered_intern_details"] = data["registered_intern_details"].rename(columns={
    "NAME": "Name",
    "COLLEGE": "College",
    "DURATION": "Duration",
    "CERTIFICATE ISSUED": "Certificate Status",
})

for name, cols in {
    "internship": ["Name", "Phone number", "College", "Duration", "Fees Status"],
    "registered_students": ["Student ID", "Student Name", "Phone Number", "College", "Duration", "Program"],
    "registered_intern_details": ["Name", "College", "Duration", "DOMAIN", "Preferred Time Slot",
                                  "Assigned Batch Code", "Batch Status", "Certificate Status"],
    "course": ["Name", "College", "Duration", "Program"],
}.items():
    data[name] = clean_text_columns(data[name], cols)

## 7. Standardise college names

The original project used a mapping dictionary to reconcile spelling, capitalisation, spacing, and abbreviation variants. Extend this mapping only when a variant has been verified; do not merge institutions based on guesswork.

In [ ]:
college_mapping = {
    "A.V.C college of engineering": "A.V.C College of Engineering",
    "Adhiparasakthi College of engineering": "Adhiparasakthi College of Engineering",
    "Jeppiaar college": "Jeppiaar Engineering College",
    "JEPPIAAR UNIVERSITY": "Jeppiaar University",
    "PANIMALAR COLLEGE": "Panimalar Engineering College",
    "Panimalar College": "Panimalar Engineering College",
    "Panimalar college": "Panimalar Engineering College",
    "Rajalakshmi engineering college": "Rajalakshmi Engineering College",
    "RMD ENGINEERING COLLEGE": "R.M.D. Engineering College",
    "RMD Engineering College": "R.M.D. Engineering College",
    "SMIT": "Sri Muthukumaran Institute of Technology",
    "Smit": "Sri Muthukumaran Institute of Technology",
    "Saveetha Engineering college": "Saveetha Engineering College",
    "saveetha Engineering college": "Saveetha Engineering College",
}

for name, frame in data.items():
    if "College" in frame.columns:
        frame["College"] = (
            frame["College"].astype("string").str.strip()
            .replace(college_mapping)
        )

## 8. Create match keys for record linkage

Names alone are not reliable identifiers. The original workflow compared cleaned name, college, and duration fields to build matching keys between internship and registration details. Review ambiguous matches before assigning identifiers.

In [ ]:
def make_match_key(df: pd.DataFrame,
                   columns: list[str],
                   separator: str = " | ") -> pd.Series:
    parts = []
    for col in columns:
        if col not in df.columns:
            raise KeyError(f"Required match column not found: {col}")
        parts.append(
            df[col].astype("string").str.strip().str.casefold().fillna("")
        )
    key = parts[0]
    for part in parts[1:]:
        key = key + separator + part
    return key.str.strip(separator).replace("", pd.NA)

# Use only where these fields exist in both tables.
match_fields = ["Name", "College", "Duration"]
if all(c in data["internship"].columns for c in match_fields) and all(
    c in data["registered_intern_details"].columns for c in match_fields
):
    data["internship"]["match_key"] = make_match_key(data["internship"], match_fields)
    data["registered_intern_details"]["match_key"] = make_match_key(
        data["registered_intern_details"], match_fields
    )
    print("Internship keys:", data["internship"]["match_key"].notna().sum())
    print("Registration-detail keys:",
          data["registered_intern_details"]["match_key"].notna().sum())
else:
    print("Check column labels before generating match keys.")

## 9. Aggregate payment transactions

The original workflow grouped payment entries by `Student ID`, calculating total payment and transaction count. Confirm the transaction amount column name before running this step.

In [ ]:
payments = data["payment_entries"].copy()

# Adjust these labels if the workbook uses different capitalisation.
payments = payments.rename(columns={
    "Student Id": "Student ID",
    "Student Name": "Student Name",
    "Amount": "Amount",
})

if {"Student ID", "Amount"}.issubset(payments.columns):
    payments["Amount"] = pd.to_numeric(payments["Amount"], errors="coerce")
    payment_summary = (
        payments.dropna(subset=["Student ID"])
        .groupby("Student ID", as_index=False)
        .agg(
            Payment_Total=("Amount", "sum"),
            Payment_Count=("Amount", "count"),
        )
    )
    display(payment_summary.head())
else:
    payment_summary = pd.DataFrame(
        columns=["Student ID", "Payment_Total", "Payment_Count"]
    )
    print("Payment summary not created: check Student ID and Amount column names.")

## 10. Reconcile collection and pending amounts

The original notebook compared recorded collections with installment amounts and payment-ledger totals, checked overpayments, and calculated pending amounts. Only apply arithmetic where the business definition is confirmed. The following helper calculates the expected pending amount without overwriting source values.

In [ ]:
def add_reconciliation_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    required = {"Total billing", "Total collection"}
    if required.issubset(out.columns):
        out["Calculated Pending"] = (
            pd.to_numeric(out["Total billing"], errors="coerce")
            - pd.to_numeric(out["Total collection"], errors="coerce")
        )
        out["Overpayment"] = (
            pd.to_numeric(out["Total collection"], errors="coerce")
            - pd.to_numeric(out["Total billing"], errors="coerce")
        ).clip(lower=0)
    else:
        print("Skipped reconciliation: expected columns not found.")
    return out

## 11. Validate duplicates, missing values, and financial inconsistencies

Run checks before deciding whether to drop or correct records. Duplicate student IDs or phone numbers may indicate data-entry issues, but they can also reflect legitimate records; investigate them rather than deleting them automatically.

In [ ]:
def duplicate_summary(df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    rows = []
    for col in columns:
        if col in df.columns:
            rows.append({
                "column": col,
                "missing": int(df[col].isna().sum()),
                "duplicate_non_null": int(df.loc[df[col].notna(), col].duplicated().sum()),
                "unique_non_null": int(df[col].nunique(dropna=True)),
            })
    return pd.DataFrame(rows)

# Choose the appropriate final/master table after the joins below.
# Example:
# duplicate_summary(final_master, ["Student ID", "Phone number", "S.No"])

## 12. Consolidation notes

The original project combined internship records with registration details and course records, enriched the master table with student IDs, and merged aggregated payment totals. The precise join order depends on the workbook's keys and column labels. Validate row counts before and after each merge, and use merge indicators to review unmatched records.

In [ ]:
def report_merge(left: pd.DataFrame, right: pd.DataFrame, *,
                 on: str | list[str], how: str = "left",
                 validate: str | None = None) -> pd.DataFrame:
    merged = left.merge(
        right, on=on, how=how, indicator=True,
        validate=validate, suffixes=("", "_right")
    )
    print("Left rows:", len(left))
    print("Right rows:", len(right))
    print("Merged rows:", len(merged))
    print("Match status:")
    print(merged["_merge"].value_counts(dropna=False))
    return merged

# Example, after confirming the relevant keys:
# final_master = report_merge(internship_master, payment_summary,
#                             on="Student ID", how="left", validate="many_to_one")
# final_master = final_master.drop(columns="_merge")

## 13. Export the cleaned dataset

Export only after completing and reviewing the merge and validation steps. The notebook intentionally does not write personal-level data automatically until `final_master` has been created and reviewed.

In [ ]:
# Uncomment after creating and validating `final_master`.
#
# output_file = PROCESSED_DIR / "student_payment_cleaned.xlsx"
# final_master.to_excel(output_file, index=False)
# print(f"Saved {len(final_master):,} rows and {len(final_master.columns):,} columns to {output_file}")

## 14. Key takeaways

- Multiple workbook sheets require consistent column names and data types before merging.
- Record linkage needs normalised matching fields and manual review of ambiguous cases.
- Payment transaction totals should be compared with reported collections and installment values.
- Duplicate and missing-value checks should inform investigation, not trigger automatic deletion.
- Sensitive student data should be anonymised or excluded before publishing project files.